# カタカナ72文字：濁点・半濁点の自動合成と学習

元の `etl5_train_resume.ipynb` をもとにした別実験です。
清音46文字の順番を維持し、濁音20文字・半濁音5文字・ヴを追加します。
**元データ・元モデルは上書きしません。上から順に実行してください。**

1. ColabのGPUランタイムを選び、画像ZIPを左のファイル欄からアップロード
2. 準備・設定・ZIP展開・データ監査を実行
3. 自動合成プレビューで点の位置や大きさを確認
4. 原画像単位で分割して学習（筆記者CSVは任意）
5. 独立したテスト分割で評価。実際の手書きでも別途確認
6. 必要ならTensorFlow.jsへ変換

既存の `processed_dataset/ｶ/*.png` のような半角フォルダ、全角フォルダの両方に対応。
手書き画像そのものは、このノートブックに同梱していません。
合成器はフォントを使わず、揺らぎのある２本の線／輪を自動描画します。
手書きの点を用意できる場合は、後述のMARK_BANKに差し替えられます。

配置v2：実際の墨線間の距離を基準に、80%を近め・20%を離れた配置にします。
接触しない範囲で位置をランダム化します。旧実験と出力・キャッシュを分離します。

In [ ]:
# 1. Colabに入っているTensorFlowを優先して使います。
# 存在しない場合だけ対応する版を取得。失敗したらこのセルで停止します。
import importlib.util, subprocess, sys
print('Python:', sys.version)
if importlib.util.find_spec('tensorflow') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'tensorflow>=2.20,<2.23'], check=True)
missing = [package for module, package in [('PIL', 'pillow'), ('matplotlib', 'matplotlib')]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 合成・学習モジュール（このセルに全コードを同梱）
"""ETL processed PNGs -> 72-class katakana model. TensorFlow is imported on demand.

Images and models stay in the user's dataset/output directory, outside this repo.
The notebook embeds this file so that it works as a single uploaded artifact.
"""
from pathlib import Path, PurePosixPath
import csv
import hashlib
import json
import math
import random
import os
import shutil
import tempfile
import time
import unicodedata
import zipfile
from contextlib import ExitStack

import numpy as np
from PIL import Image, ImageDraw, ImageOps

VERSION = "katakana-marks-v2-close-spacing"
BASE_LABELS = list("ヲアイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワン")
VOICED = dict(zip("カキクケコサシスセソタチツテトハヒフヘホウ", "ガギグゲゴザジズゼゾダヂヅデドバビブベボヴ"))
SEMIVOICED = dict(zip("ハヒフヘホ", "パピプペポ"))
LABELS = BASE_LABELS + list(VOICED.values()) + list(SEMIVOICED.values())
SIZE = 64


def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


def resolve_real_zip(path=None, upload_dir="/content"):
    """Detect capture ZIPs only; never guess which of multiple writers to use."""
    if path:
        selected = Path(path)
        if not selected.is_file():
            raise FileNotFoundError(f"実筆ZIPが見つかりません: {selected}")
    else:
        candidates = sorted(Path(upload_dir).glob("real-katakana-*.zip"))
        if not candidates:
            raise FileNotFoundError("カタカナの収集ZIPがありません。real-katakana-*.zipを/contentへアップロードするか、REAL_ZIPへ実際のパスを指定してください。ひらがなZIPはこのカタカナモデルでは評価できません。")
        if len(candidates) > 1:
            raise ValueError("実筆ZIPが複数あります。REAL_ZIPに評価する1つのパスを指定してください: " + ", ".join(map(str, candidates)))
        selected = candidates[0]
    with zipfile.ZipFile(selected) as bundle:
        if "manifest.json" in bundle.namelist():
            manifest = json.loads(bundle.read("manifest.json"))
            if manifest.get("script") not in (None, "katakana"):
                raise ValueError("このZIPはカタカナの実筆ではありません。カタカナを選んで収集してください。")
    return str(selected)


def save_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


class _ZipSet:
    """Expose several independent Drive download ZIPs as one read-only archive."""
    def __init__(self, paths):
        self.paths = paths

    def __enter__(self):
        self.stack = ExitStack()
        self.infos, self.owners = [], {}
        try:
            for index, path in enumerate(self.paths):
                bundle = self.stack.enter_context(zipfile.ZipFile(path))
                for info in bundle.infolist():
                    self.infos.append(info)
                    self.owners[id(info)] = (index, bundle)
        except BaseException:
            self.stack.close()
            raise
        return self

    def __exit__(self, *args):
        return self.stack.__exit__(*args)

    def infolist(self):
        return self.infos

    def origin(self, info):
        return self.owners[id(info)][0]

    def open(self, info):
        return self.owners[id(info)][1].open(info)


def extract_dataset_zip(zip_path, local_dir, reserve_bytes=8 * 1024**3, progress=True,
                        labels=None, require_all=True):
    """Extract class PNGs from one or several local ZIPs, preserving class folders.

    The archive is read locally; no per-image Drive operations are involved.
    Source ZIP hash isolates different datasets and permits interrupted resumes.
    """
    allowed = BASE_LABELS if labels is None else list(labels)
    paths = [zip_path] if isinstance(zip_path, (str, Path)) else list(zip_path)
    archives = sorted({Path(path).resolve() for path in paths})
    if not archives:
        raise FileNotFoundError("ZIPをすべてColabのファイル欄へアップロードしてください")
    hashes = []
    for archive in archives:
        if not archive.is_file():
            raise FileNotFoundError(f"ZIPをColabのファイル欄へアップロードしてください: {archive}")
        if progress:
            print(f"ZIPを確認しています: {archive.name} ({archive.stat().st_size/1024**2:.1f} MiB)", flush=True)
        sha = hashlib.sha256()
        with archive.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024**2), b""):
                sha.update(block)
        hashes.append(sha.hexdigest())
    archive_hash = hashes[0] if len(hashes) == 1 else digest(sorted(hashes))
    local = (Path(local_dir) / archive_hash[:16]).resolve()
    local.mkdir(parents=True, exist_ok=True)
    entries, roots, targets = [], set(), set()
    with _ZipSet(archives) as bundle:
        for info in bundle.infolist():
            name = info.filename.replace("\\", "/")
            if not info.flag_bits & 0x800:
                try:
                    decoded = name.encode("cp437").decode("cp932")
                    if any(unicodedata.normalize("NFKC", part) in allowed for part in decoded.split("/")):
                        name = decoded
                except (UnicodeError, LookupError):
                    pass
            parts = PurePosixPath(name).parts
            if name.startswith("/") or ".." in parts or any(":" in p for p in parts):
                raise ValueError(f"Invalid ZIP path: {name}")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f"ZIP symlink is not supported: {name}")
            if "__MACOSX" in parts or info.is_dir() or len(parts) < 2 or not name.lower().endswith(".png"):
                continue
            if unicodedata.normalize("NFKC", parts[-2]) not in allowed:
                continue
            roots.add((bundle.origin(info), parts[:-2]))
            relative = Path(parts[-2]) / (Path(parts[-1]).stem + ".png")
            if relative.as_posix() in targets:
                raise ValueError(f"Duplicate image in ZIP: {relative}")
            targets.add(relative.as_posix())
            entries.append((info, relative))
        if not entries:
            raise ValueError("ZIP内に『ｶ/画像.png』または『カ/画像.png』形式の画像がありません")
        if any(sum(origin == i for origin, _ in roots) > 1 for i in range(len(archives))):
            raise ValueError("ZIP内に複数のデータセットがあります。１つの画像フォルダだけをZIPにしてください")
        present = {unicodedata.normalize("NFKC", p.parent.name) for _, p in entries}
        missing = set(allowed) - present
        if missing and require_all:
            raise ValueError(f"ZIP全体で不足する文字: {sorted(missing)}。分割されたZIPをすべてアップロードしてください")
        required = sum(info.file_size for info, relative in entries if not (local / relative).is_file()
                       or (local / relative).stat().st_size != info.file_size)
        if shutil.disk_usage(local).free < required + reserve_bytes:
            raise OSError("Colabの空き容量が不足しています（展開画像と学習キャッシュ用の領域が必要です）")
        started, reused = time.monotonic(), 0
        if progress:
            print(f"ローカル展開: {len(entries):,}枚 / {sum(i.file_size for i, _ in entries)/1024**2:.1f} MiB", flush=True)
        for done, (info, relative) in enumerate(entries, 1):
            target = local / relative
            if target.is_symlink() or target.parent.is_symlink():
                raise ValueError(f"Unexpected linked destination: {target}")
            if target.is_file() and target.stat().st_size == info.file_size:
                reused += 1
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                temp = target.with_suffix(".png.part")
                if temp.is_symlink():
                    raise ValueError(f"Unexpected temporary symlink: {temp}")
                with bundle.open(info) as src, temp.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                if temp.stat().st_size != info.file_size:
                    raise OSError(f"Incomplete ZIP extraction: {relative}")
                temp.replace(target)
            if progress and (done % 1000 == 0 or done == len(entries)):
                elapsed = time.monotonic() - started
                print(f"展開 {done:,}/{len(entries):,}枚 / 経過 {elapsed:.1f}秒 / 再利用 {reused:,}枚", flush=True)
    save_json(local / ".zip-complete.json", {"sha256": archive_hash, "files": len(entries)})
    return str(local)


def stage_dataset(source_dir, local_dir, reserve_bytes=8 * 1024**3, progress=True):
    """Copy PNGs once to Colab disk, then read only local images during processing.

    Atomic per-file replacement plus preserved source mtime/size allow restart of
    interrupted copies. This is a transfer check, not a substitute for inventory's
    decoded-pixel hash. A dedicated destination is bound to one source directory.
    """
    source, local = Path(source_dir).resolve(), Path(local_dir).resolve()
    if not source.is_dir():
        raise FileNotFoundError(source)
    if source == local or source in local.parents or local in source.parents:
        raise ValueError("Copy source and destination must be separate directories")
    local.mkdir(parents=True, exist_ok=True)
    marker = local / ".source.json"
    identity = {"source": str(source)}
    if marker.exists():
        if json.loads(marker.read_text(encoding="utf-8")) != identity:
            raise ValueError("Local directory belongs to a different source")
    elif any(local.iterdir()):
        raise ValueError("Choose a new empty staging directory")
    save_json(marker, identity)
    started = time.monotonic()
    if progress:
        print("DriveのPNG一覧を取得しています…", flush=True)
    files = []
    for folder in sorted(source.iterdir()):
        if folder.is_dir() and unicodedata.normalize("NFKC", folder.name) in BASE_LABELS:
            for path in sorted(folder.glob("*.png")):
                if path.is_symlink() or source not in path.resolve().parents:
                    raise ValueError(f"Unexpected linked source: {path}")
                stat = path.stat()
                files.append((path, path.relative_to(source), stat.st_size, stat.st_mtime_ns))
    if not files:
        raise ValueError("No base-class PNGs found")
    expected = {relative.as_posix() for _, relative, _, _ in files}
    stale = [p for p in local.glob("*/*.png") if p.relative_to(local).as_posix() not in expected]
    if stale:
        raise ValueError("Source file list changed; use a new staging directory")
    # Check available space before transfer, including room for the image cache.
    remaining_bytes = sum(size for _, relative, size, mtime in files
                          if not (local / relative).is_file()
                          or (local / relative).stat().st_size != size
                          or (local / relative).stat().st_mtime_ns != mtime)
    free = shutil.disk_usage(local).free
    if free < remaining_bytes + reserve_bytes:
        raise OSError(f"Insufficient local disk: need {remaining_bytes + reserve_bytes:,} bytes, free {free:,}")
    copied, reused = 0, 0
    if progress:
        print(f"コピー対象 {len(files):,}枚 / 合計 {sum(f[2] for f in files)/1024**2:.1f} MiB", flush=True)
    for done, (path, relative, size, mtime) in enumerate(files, 1):
        target = local / relative
        if target.is_symlink() or (target.parent.exists() and target.parent.is_symlink()):
            raise ValueError(f"Unexpected linked destination: {target}")
        if target.is_file() and target.stat().st_size == size and target.stat().st_mtime_ns == mtime:
            reused += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            temp = target.with_suffix(".png.part")
            if temp.is_symlink():
                raise ValueError(f"Unexpected linked temporary file: {temp}")
            shutil.copyfile(path, temp)
            if temp.stat().st_size != size or path.stat().st_mtime_ns != mtime:
                raise OSError(f"Source changed during copy: {path}")
            os.utime(temp, ns=(mtime, mtime))
            temp.replace(target)
            copied += 1
        if progress and (done % 250 == 0 or done == len(files)):
            elapsed = time.monotonic() - started
            rate = done / max(elapsed, .001)
            print(f"{done:,}/{len(files):,}枚 | 新規 {copied:,} / 再利用 {reused:,} | "
                  f"経過 {elapsed/60:.1f}分 / 残り約 {(len(files)-done)/rate/60:.1f}分", flush=True)
    save_json(local / ".transfer.json", {"files": len(files), "copied": copied, "reused": reused,
                                        "seconds": time.monotonic()-started})
    return str(local)


def ink_image(path, polarity="auto"):
    """Return white ink on black; flatten transparent backgrounds onto white."""
    with Image.open(path) as opened:
        rgba = opened.convert("RGBA")
        image = Image.alpha_composite(Image.new("RGBA", rgba.size, "white"), rgba).convert("L")
    arr = np.asarray(image)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    if polarity not in ("auto", "white_on_black", "black_on_white"):
        raise ValueError("Unknown polarity")
    if polarity == "black_on_white" or (polarity == "auto" and np.median(border) > 127):
        image = ImageOps.invert(image)
    if not image.point(lambda p: 255 if p > 10 else 0).getbbox():
        raise ValueError(f"Blank image: {path}")
    return image


def tight(image):
    box = image.point(lambda p: 255 if p > 10 else 0).getbbox()
    if box is None:
        raise ValueError("No ink")
    return image.crop(box)


def normalize(image):
    """Browser contract: black background, white ink, bbox fit 85%, centered."""
    cropped = tight(image)
    scale = SIZE * .85 / max(cropped.size)
    # JS Math.round rather than Python banker's rounding.
    width, height = [max(1, math.floor(x * scale + .5)) for x in cropped.size]
    result = Image.new("L", (SIZE, SIZE))
    result.paste(cropped.resize((width, height), Image.Resampling.LANCZOS),
                 ((SIZE - width) // 2, (SIZE - height) // 2))
    return result


def draw_mark(kind, rng):
    """Procedural, jittered pen paths; no font or external image required."""
    scale = 4
    image = Image.new("L", (40 * scale, 40 * scale))
    draw = ImageDraw.Draw(image)
    width = rng.uniform(2.6, 4.8)
    if kind == "dakuten":
        for x in (9, 24):
            x += rng.uniform(-2, 2)
            y = rng.uniform(8, 12)
            dx, dy = rng.uniform(7, 13), rng.uniform(9, 16)
            pts = [(x, y), (x + dx * .45 + rng.uniform(-1, 1), y + dy * .5), (x + dx, y + dy)]
            pts = [(round(a * scale), round(b * scale)) for a, b in pts]
            w = round(width * scale)
            draw.line(pts, fill=255, width=w, joint="curve")
            for a, b in (pts[0], pts[-1]):
                draw.ellipse((a-w/2, b-w/2, a+w/2, b+w/2), fill=255)
    elif kind == "handakuten":
        rx, ry = rng.uniform(9, 13), rng.uniform(9, 13)
        phase = rng.uniform(0, 2 * math.pi)
        pts = []
        for i in range(65):
            angle = 2 * math.pi * i / 64
            wobble = 1 + .045 * math.sin(3 * angle + phase)
            pts.append(((20 + rx * math.cos(angle) * wobble) * scale,
                        (20 + ry * math.sin(angle) * wobble) * scale))
        draw.line(pts, fill=255, width=round(width * scale), joint="curve")
    else:
        raise ValueError(kind)
    image = image.rotate(rng.uniform(-18, 18), resample=Image.Resampling.BICUBIC)
    return tight(image).resize((28, 28), Image.Resampling.LANCZOS)


def dilate_ink(mask, radius):
    """Square dilation in O(pixels), without a SciPy/Colab dependency."""
    padded = np.pad(mask.astype(np.int32), radius)
    sums = np.pad(padded, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    size = 2 * radius + 1
    return (sums[size:, size:] - sums[:-size, size:]
            - sums[size:, :-size] + sums[:-size, :-size]) > 0


def place_mark(canvas, mark, bounds, rng):
    """Choose by actual ink distance, allowing overlapping empty bounding boxes.

    80% use a 3..6 pixel gap, 20% 7..11, on the 144px working canvas.
    Gap is Chebyshev distance between ink pixels, before final normalization.
    """
    x, y, bw, bh = bounds
    mw, mh = mark.size
    width, height = canvas.size
    gap = rng.randint(3, 6) if rng.random() < .8 else rng.randint(7, 11)
    ink = np.asarray(canvas) > 10
    mark_ink = np.asarray(mark) > 10
    forbidden = dilate_ink(ink, gap - 1)
    nearby = dilate_ink(ink, gap + 1)
    # Keep marks in the upper-right neighbourhood; let them enter the empty
    # space beside slanted/curved strokes instead of requiring an empty box.
    positions = [(mx, my)
                 for my in range(max(2, y - mh - 12), min(height - 2 - mh, y + bh // 3) + 1)
                 for mx in range(max(2, x + bw - mw - 16), min(width - 2 - mw, x + bw + 12) + 1)]
    rng.shuffle(positions)
    for mx, my in positions:
        region = np.s_[my:my + mh, mx:mx + mw]
        if not np.any(forbidden[region] & mark_ink) and np.any(nearby[region] & mark_ink):
            return mx, my, gap
    raise ValueError("Cannot place mark near upper-right ink without contact")


def compose(image, kind=None, seed=0, layout="natural", mark_path=None):
    """Place mark by the upper right of base ink; normalize the *whole* character.

    Alternate layouts also apply to clean letters to reduce a base-size shortcut.
    A hand-drawn donor (if provided) replaces the procedural mark.
    """
    rng = random.Random(seed)
    base = tight(image)
    canvas = Image.new("L", (144, 144))
    target = rng.randint(81, 88) if layout == "natural" else rng.randint(70, 82)
    scale = target / max(base.size)
    bw, bh = [max(1, round(d * scale)) for d in base.size]
    # Reserve enough height for even a wide top stroke (e.g. コ) plus the mark.
    x, y = rng.randint(8, 14), rng.randint(44, 46)
    canvas.paste(base.resize((bw, bh), Image.Resampling.LANCZOS), (x, y))
    if kind:
        mark = tight(ink_image(mark_path)) if mark_path else draw_mark(kind, rng)
        mw = rng.randint(18, 26)
        mh = max(8, round(mark.height * mw / mark.width))
        mh = min(mh, 28)
        mark = mark.resize((mw, mh), Image.Resampling.LANCZOS)
        mx, my, _ = place_mark(canvas, mark, (x, y, bw, bh), rng)
        # Empty corners of the mark's rectangle must not erase base strokes.
        region = np.asarray(canvas.crop((mx, my, mx + mw, my + mh)))
        canvas.paste(Image.fromarray(np.maximum(region, np.asarray(mark))), (mx, my))
    return normalize(canvas)


def inventory(dataset_dir, groups_csv=None, seed=20261016, polarity="auto", confirm_originals=False, progress=False):
    """Group original and augmented siblings before splitting, never after synthesis.

    CSV columns: path,group. path is relative to dataset_dir, group is writer ID
    (preferred) or original-record ID. A missing CSV only deduplicates exact pixels;
    transformed siblings cannot be inferred reliably from processed PNGs.
    """
    root = Path(dataset_dir)
    groups = None
    if groups_csv:
        groups = {}
        with open(groups_csv, encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                key = row["path"].replace("\\", "/")
                if key in groups or not row["group"].strip():
                    raise ValueError(f"Invalid/duplicate group row: {key}")
                groups[key] = row["group"].strip()
    dirs = {}
    for folder in root.iterdir():
        if folder.is_dir():
            name = unicodedata.normalize("NFKC", folder.name)
            if name in BASE_LABELS:
                if name in dirs:
                    raise ValueError(f"Duplicate class directories: {name}")
                dirs[name] = folder
    missing = set(BASE_LABELS) - dirs.keys()
    if missing:
        raise ValueError(f"Missing base classes: {sorted(missing)}")
    if progress:
        print("PNGのファイル一覧を取得しています（Driveではこの処理にも時間がかかります）", flush=True)
    files = {char: sorted(dirs[char].glob("*.png")) for char in BASE_LABELS}
    total = sum(len(paths) for paths in files.values())
    started, done = time.monotonic(), 0
    if progress:
        print(f"監査開始: {total:,}枚。GPUは使わず、画像を１枚ずつ読み込みます。", flush=True)
    rows, hashes, duplicates = [], {}, 0
    for char in BASE_LABELS:
        if progress:
            print(f"{char}: {len(files[char]):,}枚 / 全体 {done:,}/{total:,}", flush=True)
        for path in files[char]:
            relative = path.relative_to(root).as_posix()
            image = ink_image(path, polarity)
            h = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
            done += 1
            if progress and (done % 250 == 0 or done == total):
                elapsed = time.monotonic() - started
                rate = done / max(elapsed, .001)
                remaining = (total - done) / rate
                print(f"  {done:,}/{total:,} ({done/total:.1%}) | "
                      f"経過 {elapsed/60:.1f}分 | {rate:.1f}枚/秒 | 残り約 {remaining/60:.1f}分", flush=True)
            if groups is not None and relative not in groups:
                raise ValueError(f"Missing group ID: {relative}")
            group = groups[relative] if groups is not None else h
            if h in hashes:
                previous = hashes[h]
                if previous[0] != char or previous[1] != group:
                    raise ValueError(f"Conflicting duplicate image/group: {relative}")
                duplicates += 1
                continue
            hashes[h] = (char, group)
            # Whole writer groups remain together across character classes.
            value = int(hashlib.sha256(f"{seed}:{group}".encode()).hexdigest()[:8], 16) / 2**32
            split = "train" if value < .8 else "val" if value < .9 else "test"
            rows.append(dict(path=relative, base=char, group=group, split=split, sha256=h))
    for char in BASE_LABELS:
        for split in ("train", "val", "test"):
            if not any(r["base"] == char and r["split"] == split for r in rows):
                raise ValueError(f"No {split} source for {char}; more independent groups needed")
    grouping = "provided" if groups is not None else "confirmed_original_records" if confirm_originals else "exact_pixels_only"
    return rows, {"grouping": grouping,
                  "removed_exact_duplicates": duplicates, "source_count": len(rows)}


def make_examples(sources, variants=2, seed=20261016):
    if variants < 1:
        raise ValueError("variants must be >= 1")
    examples = []
    for source in sources:
        targets = [(source["base"], None)]
        if source["base"] in VOICED:
            targets.append((VOICED[source["base"]], "dakuten"))
        if source["base"] in SEMIVOICED:
            targets.append((SEMIVOICED[source["base"]], "handakuten"))
        for label, kind in targets:
            for variant in range(variants):
                row_seed = int(digest([seed, source["sha256"], label, variant])[:12], 16)
                examples.append({**source, "label": label, "kind": kind, "seed": row_seed,
                                 "layout": "natural" if variant % 2 == 0 else "compact"})
    return examples


def render_example(row, root, polarity="auto", mark_bank=None):
    donor = None
    if row["kind"] and mark_bank:
        donors = sorted((Path(mark_bank) / row["kind"] / row["split"]).glob("*.png"))
        if not donors:
            raise ValueError(f"No independent {row['split']} mark donors for {row['kind']}")
        donor = donors[row["seed"] % len(donors)]
    return compose(ink_image(Path(root) / row["path"], polarity), row["kind"],
                   row["seed"], row["layout"], donor)


def preview(examples, root, output, polarity="auto", mark_bank=None):
    """Rows are base / voiced / semi-voiced. Labels are in an adjacent JSON index."""
    selected = []
    for base in VOICED:
        labels = [base, VOICED[base], SEMIVOICED.get(base)]
        selected.append([next((r for r in examples if r["label"] == label), None) for label in labels])
    cell = 144
    sheet = Image.new("RGB", (len(selected) * cell, 3 * cell), "#d8d8d8")
    index = []
    for col, triplet in enumerate(selected):
        for row_no, row in enumerate(triplet):
            if row:
                im = render_example(row, root, polarity, mark_bank).resize((128, 128))
                sheet.paste(im.convert("RGB"), (col * cell + 8, row_no * cell + 8))
                index.append({"column": col, "row": row_no, "label": row["label"], "source": row["path"]})
    sheet.save(output)
    save_json(Path(output).with_suffix(".json"), index)
    return sheet


def dataset(examples, root, split, batch=128, polarity="auto", mark_bank=None, cache_file=None):
    import tensorflow as tf
    rows = [row for row in examples if row["split"] == split]
    if split == "train":
        # Avoid batches dominated by one class even when the local shuffle buffer
        # is smaller than a class. This order is also what gets disk-cached.
        random.Random(20261016).shuffle(rows)

    def generate():
        for row in rows:
            image = render_example(row, root, polarity, mark_bank)
            yield np.asarray(image, dtype=np.float32)[..., None] / 255., np.int32(LABELS.index(row["label"]))

    ds = tf.data.Dataset.from_generator(generate, output_signature=(
        tf.TensorSpec((SIZE, SIZE, 1), tf.float32), tf.TensorSpec((), tf.int32)))
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(rows)))
    if cache_file:
        # Store the first generated pass on local Colab disk, not Google Drive.
        ds = ds.cache(str(cache_file))
    if split == "train":
        ds = ds.shuffle(min(4096, len(rows)), seed=20261016, reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(1)


def build_model(baseline=None):
    import tensorflow as tf
    layers = tf.keras.layers
    model = tf.keras.Sequential([
        layers.Input((64, 64, 1)),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(), layers.Dropout(.25),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(), layers.Dropout(.25),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(), layers.Dropout(.25),
        layers.Flatten(), layers.Dense(512, activation="relu"), layers.Dropout(.25),
        layers.Dense(len(LABELS), activation="softmax")])
    if baseline:
        old = tf.keras.models.load_model(baseline, compile=False)
        if old.input_shape[1:] != (64, 64, 1) or old.output_shape[-1] != 46:
            raise ValueError("Baseline must be the original 64x64 / 46-class model")
        old_layers = [layer for layer in old.layers if layer.weights]
        new_layers = [layer for layer in model.layers if layer.weights]
        if len(old_layers) != len(new_layers):
            raise ValueError("Baseline architecture mismatch")
        for a, b in zip(old_layers[:-1], new_layers[:-1]):
            b.set_weights(a.get_weights())
        kernel, bias = new_layers[-1].get_weights()
        old_kernel, old_bias = old_layers[-1].get_weights()
        kernel[:, :46], bias[:46] = old_kernel, old_bias
        new_layers[-1].set_weights([kernel, bias])
    model.compile(optimizer=tf.keras.optimizers.Adam(2e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def train(examples, dataset_dir, output_dir, metadata, epochs=100, batch=128,
          baseline=None, polarity="auto", mark_bank=None, allow_unverified_groups=False):
    import tensorflow as tf
    if metadata["grouping"] not in ("provided", "confirmed_original_records") and not allow_unverified_groups:
        raise ValueError("Original/writer groups are unknown. Preview works, but training is stopped. "
                         "Provide groups.csv, or explicitly allow a diagnostic run with possible leakage.")
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    contract = dict(version=VERSION, labels=LABELS, examples_digest=digest(examples),
                    metadata=metadata, polarity=polarity, batch=batch,
                    mark_bank=str(mark_bank), seed=20261016)
    # Donor content changes also invalidate a resumed experiment.
    if mark_bank:
        contract["donors"] = {p.relative_to(mark_bank).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                              for p in sorted(Path(mark_bank).rglob("*.png"))}
        donor_splits = {}
        for relative, sha in contract["donors"].items():
            parts = Path(relative).parts
            if len(parts) != 3 or parts[0] not in ("dakuten", "handakuten") or parts[1] not in ("train", "val", "test"):
                raise ValueError(f"Unexpected mark-bank path: {relative}")
            donor_splits.setdefault(sha, set()).add(parts[1])
        if any(len(splits) > 1 for splits in donor_splits.values()):
            raise ValueError("Identical mark donor appears in multiple splits")
    if baseline:
        contract["baseline_sha256"] = hashlib.sha256(Path(baseline).read_bytes()).hexdigest()
    config_path = out / "experiment.json"
    if config_path.exists() and json.loads(config_path.read_text(encoding="utf-8")) != contract:
        raise ValueError("Output directory belongs to a different experiment; choose a new directory")
    save_json(config_path, contract)
    save_json(out / "labels.json", LABELS)
    (out / "katakana_labels_full.txt").write_text("".join(LABELS), encoding="utf-8")
    (out / "labels.js").write_text("const LABELS_KATAKANA = " + json.dumps(LABELS, ensure_ascii=False) + ";\n", encoding="utf-8")
    save_json(out / "examples.json", examples)
    state_path = out / "state.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {
        "epoch": 0, "best": -1., "wait": 0}
    last = out / f"epoch-{state['epoch']:04d}.keras"
    model = tf.keras.models.load_model(last) if state["epoch"] else build_model(baseline)
    cache_dir = Path(tempfile.gettempdir()) / "katakana-cache" / digest(contract)
    cache_dir.mkdir(parents=True, exist_ok=True)
    train_ds = dataset(examples, dataset_dir, "train", batch, polarity, mark_bank, cache_dir / "train")
    val_ds = dataset(examples, dataset_dir, "val", batch, polarity, mark_bank, cache_dir / "val")

    class SaveEpoch(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            score = float(logs["val_accuracy"])
            if not math.isfinite(score):
                raise ValueError("Non-finite validation accuracy")
            improved = score > state["best"]
            new_state = {"epoch": epoch+1, "best": max(state["best"], score),
                         "wait": 0 if improved else state["wait"]+1}
            # Immutable epoch file; state commits only after complete model save.
            current = out / f"epoch-{epoch+1:04d}.keras"
            self.model.save(current)
            if improved:
                new_state["best_epoch"] = epoch+1
            else:
                new_state["best_epoch"] = state["best_epoch"]
            save_json(state_path, new_state)
            state.update(new_state)
            if state["wait"] >= 20:
                self.model.stop_training = True

    if state["wait"] < 20 and state["epoch"] < epochs:
        model.fit(train_ds, validation_data=val_ds, epochs=epochs, initial_epoch=state["epoch"],
                  callbacks=[SaveEpoch(), tf.keras.callbacks.CSVLogger(str(out / "history.csv"), append=True)])
    if "best_epoch" not in state:
        raise ValueError("No completed epoch")
    best_path = out / f"epoch-{state['best_epoch']:04d}.keras"
    print("Best checkpoint:", best_path)
    return tf.keras.models.load_model(best_path)


def evaluate(model, examples, root, output, polarity="auto", mark_bank=None, baseline=None):
    """Held-out *synthetic* test. Never describe it as real voiced-handwriting accuracy."""
    ds = dataset(examples, root, "test", polarity=polarity, mark_bank=mark_bank)
    true, probs, old_probs = [], [], []
    for images, labels in ds:
        true.extend(labels.numpy().tolist())
        probs.extend(model(images, training=False).numpy())
        if baseline is not None:
            old_probs.extend(baseline(images, training=False).numpy())
    probs, true = np.asarray(probs), np.asarray(true)
    ids = probs.argmax(axis=1)
    top5 = np.argsort(probs, axis=1)[:, -5:]
    cm = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    np.add.at(cm, (true, ids), 1)
    result = {"evaluation": "held-out sources with synthetic marks; not real handwriting accuracy",
              "top1": float(np.mean(ids == true)), "top5": float(np.mean((top5 == true[:, None]).any(axis=1))),
              "characters": {}, "confusion": cm.tolist()}
    for i, char in enumerate(LABELS):
        mask = true == i
        result["characters"][char] = {"count": int(mask.sum()), "top1": float(np.mean(ids[mask] == i))}
    if baseline is not None:
        mask = true < 46
        result["clean_same_inputs"] = {
            "baseline": float(np.mean(np.asarray(old_probs).argmax(axis=1)[mask] == true[mask])),
            "new": float(np.mean(ids[mask] == true[mask]))}
    save_json(output, result)
    return result

In [ ]:
# 3. 設定：元モデルとは別の出力先です。
# DriveのダウンロードでZIPが複数になった場合も、全部そのままアップロード。
ZIP_PATHS = [p for p in sorted(Path('/content').glob('*.zip')) if not p.name.startswith('real-')]
# 実筆テスト用のreal-*.zipは学習用データに混ぜません。
# 無関係なZIPがある場合: ZIP_PATHS = ['/content/part1.zip', '/content/part2.zip', ...]
LOCAL_DATASET_DIR = '/content/katakana-zip-local'
OUTPUT_DIR = '/content/drive/MyDrive/ETL/model_katakana_marks_v2_close_spacing'
BASELINE_MODEL = '/content/drive/MyDrive/ETL/model_katakana/model.h5'
# 元モデルがない場合だけ None に変更（最初から学習）。

GROUPS_CSV = None  # 例: '/content/drive/MyDrive/ETL/groups.csv'
# ユーザー確認: 元からある画像を使うため、原画像ごとの分割を標準にします。
# 別途増幅した画像を混ぜる場合は、GROUPS_CSVに元画像のIDを設定してください。
CONFIRM_ONE_PNG_PER_ORIGINAL = True
MARK_BANK = None   # 手書き点のフォルダ。Noneなら自動描画。
POLARITY = 'auto'  # auto / white_on_black / black_on_white
VARIANTS = 2       # 各原画像・各文字の合成バリエーション数
EPOCHS = 100
BATCH_SIZE = 128
SEED = 20261016
# 原画像の由来が不明なまま、合成動作を試す実験に限りTrue。
# その成績を独立した認識率として扱わないでください。
ALLOW_UNVERIFIED_GROUPS = False

if BASELINE_MODEL and not Path(BASELINE_MODEL).is_file():
    raise FileNotFoundError(f'元モデルが見つかりません: {BASELINE_MODEL}')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('72文字:', ''.join(LABELS))

In [ ]:
# 3.5. アップロード済みZIPをColab内で展開。DriveのPNGは読みません。
# Colab左のフォルダアイコン → ファイルをアップロード → 分割されたZIPをすべて選択。
# 『processed_dataset/ｶ/*.png』でも『ｶ/*.png』でも自動で認識します。
# ZIPの中身は、元データまたは前処理済みの文字別PNGフォルダにしてください。
# 再実行では同じZIPの展開済みファイルを再利用します。
print('使用するZIP:', [str(path) for path in ZIP_PATHS])
DATASET_DIR = extract_dataset_zip(ZIP_PATHS, LOCAL_DATASET_DIR)
print('以後の画像読込先:', DATASET_DIR)

### 学習用と評価用の混入を防ぐ設定

元ノートは各文字2,800枚を使っていますが、これらが元画像を増幅したものなら、
単純にファイルを分けても同じ原画像の派生が両方に入り、成績が高く出ます。

`groups.csv` の例（pathはDATASET_DIRからの相対パス）:

```csv
path,group
ｶ/original0001.png,writer001
ｶ/original0001_rotate.png,writer001
ｷ/original0002.png,writer001
ｶ/original0003.png,writer002
```

**標準では筆記者CSVなしで、そのまま学習できます。**
元からある画像を使うという確認に基づき、`CONFIRM_ONE_PNG_PER_ORIGINAL = True` にしています。
原画像を先に学習・検証・テストへ分け、それから濁点・半濁点を合成します。
同じ原画像からできた清音・濁音・半濁音が、別の分割へ混ざることはありません。
完全一致の重複画像も除外します。

筆記者IDは、未知の筆記者に対する性能を別途測りたい場合の任意設定です。
その場合はCSVのgroupに筆記者IDを指定します。
すでに回転・拡縮などで増幅した画像を入力する場合も、CSVで同じ原画像の派生をまとめてください。
添付の `etl5_preprocess.ipynb` 自体は、切り抜き・拡縮だけで枚数を増やしていません。

任意の手書き点は `MARK_BANK/dakuten/train/*.png`、`dakuten/val/*.png`、
`dakuten/test/*.png`（handakutenも同様）に配置します。
点も原画像・筆記者を分け、別の分割へ同じ点をコピーしないでください。

In [ ]:
# 4. データ監査 → 原画像グループの分割 → 合成対象リスト
from collections import Counter
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
sources, audit = inventory(DATASET_DIR, GROUPS_CSV, SEED, POLARITY, CONFIRM_ONE_PNG_PER_ORIGINAL,
                           progress=True)
print('画像監査完了。合成対象リストを作っています…', flush=True)
examples = make_examples(sources, VARIANTS, SEED)
print(audit)
print('元画像:', Counter(r['split'] for r in sources))
print('合成後:', Counter(r['split'] for r in examples))
save_json(Path(OUTPUT_DIR) / 'source-audit.json', audit)

In [ ]:
# 5. 合成プレビュー：上段=清音、中段=濁音、下段=半濁音
from IPython.display import display
sheet = preview(examples, DATASET_DIR, Path(OUTPUT_DIR) / 'preview.png', POLARITY, MARK_BANK)
display(sheet)
print('列:', ' '.join(VOICED))
print('濁音:', ' '.join(VOICED.values()))
print('点の近さ・接触・欠落を確認してください。近い配置と離れた配置を混ぜています。preview.jsonに文字と元画像を保存します。')

In [ ]:
# 6. 学習：同じ設定・データなら、完了した最後のエポックから再開します。
# 初回は合成画像をローカルディスクにキャッシュします（規模により数GB）。
# ２エポック目以降は画像の再読込・再合成を省きます。
# 特徴抽出層と元の46出力を引き継ぎ、全層を再学習します。
# 過去のモデルのラベル順がBASE_LABELSと一致することが前提です。
best_model = train(examples, DATASET_DIR, OUTPUT_DIR, audit,
                   epochs=EPOCHS, batch=BATCH_SIZE, baseline=BASELINE_MODEL,
                   polarity=POLARITY, mark_bank=MARK_BANK,
                   allow_unverified_groups=ALLOW_UNVERIFIED_GROUPS)

In [ ]:
# 7. テストはモデル選択に使っていない分割。
# 濁音のテスト画像も合成なので、実際の子どもの筆跡での精度とは別です。
baseline = tf.keras.models.load_model(BASELINE_MODEL, compile=False) if BASELINE_MODEL else None
result = evaluate(best_model, examples, DATASET_DIR, Path(OUTPUT_DIR) / 'test-report.json',
                  POLARITY, MARK_BANK, baseline)
print('合成テスト Top1 / Top5:', result['top1'], result['top5'])
print('清音・同じ入力で比較:', result.get('clean_same_inputs'))
for char, stats in result['characters'].items():
    print(char, stats)

### 実筆で確認してからアプリへ反映

`ガ`を実際に書いた画像と、清音に点を合成した画像は違います。
今回のテスト結果だけでは、実筆での認識率向上は判断できません。
指・ペン・マウスの実筆、点の位置が離れた文字、シ／ツ／ソ／ン、清音への誤った濁点付与を確認してください。

下の任意セルで、黒背景白文字／白背景黒文字の実筆PNGを
`REAL_DIR/ガ/*.png` のようにまとめて評価できます。
各画像は１文字。人が確認した正解フォルダに入れてください。
このデータを後から追加学習に使う場合、別の実筆テストを確保してください。

In [ ]:
# 8. 合成ではない実筆の評価。収集アプリのZIPをアップロードして実行。
REAL_ZIP = None  # Noneなら/contentのreal-katakana-*.zipを自動検出。複数ある場合は1つのパスを指定。
REAL_DIR = None  # '/content/drive/MyDrive/ETL/real_katakana_test'
print('実筆評価を開始します。入力データを確認しています。', flush=True)
if not REAL_DIR or REAL_ZIP:
    REAL_ZIP = resolve_real_zip(REAL_ZIP)
if REAL_ZIP:
    print('使用する実筆ZIP:', REAL_ZIP, flush=True)
    REAL_DIR = extract_dataset_zip(REAL_ZIP, '/content/katakana-real-test', reserve_bytes=0,
                                   labels=LABELS, require_all=False)
if REAL_DIR:
    real = []
    for folder in sorted(Path(REAL_DIR).iterdir()):
        if folder.is_dir() and folder.name in LABELS:
            for path in sorted(folder.glob('*.png')):
                real.append((folder.name, path))
    if not real:
        raise ValueError('実筆画像がありません')
    print(f'実筆画像 {len(real)}枚・{len(set(char for char, _ in real))}文字を評価します。', flush=True)
    if 'best_model' not in globals():
        raise RuntimeError('評価モデルが未読込です。先に⑥の学習セルを実行してbest_modelを準備してください。')
    stats = {char: {'count': 0, 'top1': 0, 'top5': 0} for char in LABELS}
    for char, path in real:
        arr = np.asarray(normalize(ink_image(path)), dtype=np.float32)[None, ..., None] / 255.
        probs = best_model(arr, training=False).numpy()[0]
        ids = np.argsort(probs)
        entry = stats[char]
        entry['count'] += 1
        entry['top1'] += int(LABELS[ids[-1]] == char)
        entry['top5'] += int(LABELS.index(char) in ids[-5:])
    report = {'evaluation': 'real handwriting', 'characters': stats,
              'missing_characters': [c for c, s in stats.items() if not s['count']]}
    save_json(Path(OUTPUT_DIR) / 'real-test-report.json', report)
    total = len(real)
    top1 = sum(s['top1'] for s in stats.values()) / total
    top5 = sum(s['top5'] for s in stats.values()) / total
    print(f'実筆評価完了: {total}枚 / Top1 {top1:.1%} / Top5 {top5:.1%}')
    for char, entry in stats.items():
        if entry['count']:
            print(char, entry)
    print('未評価の文字:', ''.join(report['missing_characters']))
    print('結果保存先:', Path(OUTPUT_DIR) / 'real-test-report.json')

### TensorFlow.jsへの変換（任意、学習完了後）

現アプリのカタカナと同じGraphModel形式です。
コンバータは別のPython環境にインストールし、学習中のTensorFlowを変更しません。
変換用はPython 3.12を使い、Colab本体のPython版から切り離します。
依存関係の取得には時間と空き容量が必要です。
元の46文字モデルの出力先は変更しません。

変換結果を使う際は `model.json` と重みファイルに加え、
**`labels.json` の72文字順をアプリ側に反映する必要があります。**
このノートブックはアプリや公開モデルを自動で置き換えません。

In [ ]:
# 9. SavedModelを別フォルダへ保存。既存の出力は削除しません。
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f')
saved = Path(OUTPUT_DIR) / ('saved_model-' + stamp)
best_model.export(str(saved))

In [ ]:
# 10. 独立した変換環境を用意（初回のみインストール）。失敗は例外で停止。
import subprocess, sys

def run_conversion_command(command, log_name, env=None):
    # 子プロセスの本当のエラーを、CalledProcessErrorに埋もれさせない。
    log_path = Path(OUTPUT_DIR) / log_name
    print('実行:', ' '.join(map(str, command)), flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', env=env) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            code = process.wait()
    print('ログ保存先:', log_path, flush=True)
    if code:
        raise RuntimeError(f'変換の準備または実行が終了コード{code}で失敗しました。直前のログ末尾20〜40行を確認してください。全ログ: {log_path}')

converter_env = Path('/content/katakana-tfjs-env')
if not (converter_env / 'bin/python').exists():
    # 2.19のコンバータ環境をPython 3.13以降に作らないよう、3.12を明示。
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12',
                    '--seed', str(converter_env)], check=True)
converter_python = str(converter_env / 'bin/python')
subprocess.run([converter_python, '-c',
                'import sys; assert sys.version_info[:2] == (3, 12), "変換用環境を新しいパスで作り直してください"'], check=True)
run_conversion_command([converter_python, '-m', 'pip', 'install',
                'tensorflowjs==4.22.0', 'tensorflow==2.19.0', 'tf-keras==2.19.0'], 'converter-install.log')
run_conversion_command([converter_python, '-c',
    'import sys, importlib.metadata as m; print(sys.version); '
    'print({p: m.version(p) for p in ["tensorflowjs", "tensorflow", "tf-keras", "keras", "numpy"]})'],
    'converter-versions.log')
tfjs_out = Path(OUTPUT_DIR) / ('tfjs-' + stamp)
run_conversion_command([str(converter_env / 'bin/tensorflowjs_converter'),
                '--input_format=tf_saved_model', '--output_format=tfjs_graph_model',
                str(saved), str(tfjs_out)], 'converter-' + stamp + '.log')
converted = json.loads((tfjs_out / 'model.json').read_text())
assert converted['format'] == 'graph-model'
for group in converted['weightsManifest']:
    for filename in group['paths']:
        assert (tfjs_out / filename).is_file(), filename
save_json(tfjs_out / 'labels.json', LABELS)
save_json(tfjs_out / 'preprocessing.json', {
    'size': 64, 'channels': 1, 'range': [0, 1], 'ink': 'white',
    'background': 'black', 'bbox_fit': .85, 'classes': len(LABELS),
    'note': 'Browser interpolation and Pillow interpolation are not pixel-identical.'})
print('変換完了:', tfjs_out)
print('ラベルも同時に更新してください:', ''.join(LABELS))